<a href="https://colab.research.google.com/github/Abdalgne/HAPI-sinc-SQLite/blob/main/FHIR-SQLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== CÉLULA 1: INSTALAÇÃO DE DEPENDÊNCIAS =====
# Esta célula instala as bibliotecas necessárias para o funcionamento do código

import subprocess
import sys

# Instala a biblioteca 'requests' para fazer requisições HTTP ao servidor FHIR
!pip install requests -q

print("✅ Dependências instaladas com sucesso!")

✅ Dependências instaladas com sucesso!


In [ ]:
# ===== CÉLULA 2: IMPORTAÇÕES E CONFIGURAÇÕES =====
# Importa todas as bibliotecas necessárias para o funcionamento da interface

import requests
import sqlite3
from datetime import datetime
import json
import os
from typing import List, Dict, Optional

# ===== CONFIGURAÇÕES GLOBAIS =====
# URL do servidor FHIR que usaremos para armazenar e buscar dados
FHIR_SERVER_URL = "https://hapi.fhir.org/baseR4"

# Nome do arquivo do banco de dados SQLite
DATABASE_NAME = "b214_fhir.db"

print("✅ Importações realizadas com sucesso!")

✅ Importações realizadas com sucesso!


In [ ]:
# ===== CÉLULA 3: FUNÇÕES DO BANCO DE DADOS =====
# Funções para criar, conectar e gerenciar o banco de dados SQLite

def criar_banco_dados():
    """
    Descrição: Cria o banco de dados SQLite com as tabelas necessárias
    Parâmetros: Nenhum
    Retorno: Conexão com o banco de dados
    """
    # Estabelece conexão com o banco de dados
    conexao = sqlite3.connect(DATABASE_NAME)
    # pega uma "caneta" para mexer com os dados
    cursor = conexao.cursor()

    # ===== CRIAÇÃO DA TABELA PATIENTS =====
    # Tabela para armazenar informações de pacientes
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            fhir_id TEXT UNIQUE NOT NULL,
            family_name TEXT NOT NULL,
            given_name TEXT NOT NULL,
            gender TEXT,
            birth_date TEXT,
            phone TEXT,
            city TEXT,
            state TEXT,
            postal_code TEXT
        )
    ''')

    # ===== CRIAÇÃO DA TABELA OBSERVATIONS =====
    # Tabela para armazenar observações de pacientes (sinais vitais)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS observations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            fhir_id TEXT UNIQUE NOT NULL,
            patient_fhir_id TEXT NOT NULL,
            observation_type TEXT NOT NULL,
            value REAL,
            unit TEXT,
            effective_date_time TEXT,
            FOREIGN KEY(patient_fhir_id) REFERENCES patients(fhir_id)
        )
    ''')

    # Confirma/salva as alterações no banco de dados
    conexao.commit()
    print(f"✅ Banco de dados '{DATABASE_NAME}' criado/verificado com sucesso!")

    return conexao

def obter_conexao():
    """
    Descrição: Obtém uma conexão com o banco de dados existente
    Parâmetros: Nenhum
    Retorno: Conexão com o banco de dados
    """
    return sqlite3.connect(DATABASE_NAME)

# Cria o banco de dados na inicialização
conexao_inicial = criar_banco_dados()
conexao_inicial.close()

✅ Banco de dados 'b214_fhir.db' criado/verificado com sucesso!


In [ ]:
# ===== CÉLULA 4: FUNÇÕES DE REQUISIÇÃO AO FHIR =====
# Funções para comunicação com o servidor HAPI FHIR

def buscar_pacientes_por_nome(nome: str) -> List[Dict]:
    """
    Descrição: Busca pacientes no FHIR pelo nome
    Parâmetros: nome (string) - Nome do paciente a buscar
    Retorno: Lista de dicionários com dados dos pacientes encontrados
    """
    try:
        # Monta a URL de busca com o nome do paciente
        url = f"{FHIR_SERVER_URL}/Patient?name={nome}"
        # Faz a requisição GET ao servidor FHIR
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()

        dados = resposta.json()
        pacientes = []

        # Extrai os pacientes da resposta
        if 'entry' in dados:
            for entrada in dados['entry']:
                recurso = entrada['resource']
                pacientes.append(recurso)

        return pacientes
    except Exception as e:
        print(f"❌ Erro ao buscar pacientes: {e}")
        return []

"""
O FHIR retorna algo assim:

 {
     "resourceType": "Bundle",
     "entry": [
         {
             "resource": {
                 "resourceType": "Patient",
                 "id": "123",
                 "name": [{"family": "Silva", "given": ["João"]}]
             }
         },
         {
             "resource": {
                 "resourceType": "Patient",
                 "id": "456",
                 "name": [{"family": "Silva", "given": ["Maria"]}]
             }
         }
     ]
 }


A lista de pacientes fica mais ou menos assim:

 [
     {
         'resourceType': 'Patient',
         'id': '123',
         'name': [{'family': 'Silva', 'given': ['João']}],
         ...
     },
     {
         'resourceType': 'Patient',
         'id': '456',
         'name': [{'family': 'Silva', 'given': ['Maria']}],
         ...
     }
 ]

"""


def criar_paciente_fhir(family_name: str, given_name: str, gender: str,
                        birth_date: str, phone: str, city: str,
                        state: str, postal_code: str) -> Optional[str]:
    """
    OBS: ->            "Esta função retorna..."
         Optional[str] "...uma string OU None"

    Descrição: Cria um novo paciente no servidor FHIR
    Parâmetros: family_name, given_name, gender, birth_date, phone, city, state, postal_code
    Retorno: ID do paciente criado no FHIR (string) ou None se erro
    """
    try:
        # Monta o corpo da requisição seguindo o padrão FHIR
        payload = {
            "resourceType": "Patient",
            "name": [
                {
                    "use": "official",
                    "family": family_name,
                    "given": [given_name]
                }
            ],
            "gender": gender,
            "birthDate": birth_date,
            "telecom": [
                {
                    "system": "phone",
                    "value": phone
                }
            ],
            "address": [
                {
                    "city": city,
                    "state": state,
                    "postalCode": postal_code
                }
            ]
        }

        # Faz a requisição POST para criar o paciente
        url = f"{FHIR_SERVER_URL}/Patient"
        resposta = requests.post(url, json=payload, timeout=10)
        resposta.raise_for_status()

        dados = resposta.json()
        fhir_id = dados.get('id')
        print(f"✅ Paciente criado no FHIR com ID: {fhir_id}")
        return fhir_id
    except Exception as e:
        print(f"❌ Erro ao criar paciente no FHIR: {e}")
        return None

def buscar_paciente_por_id_fhir(fhir_id: str) -> Optional[Dict]:
    """
    Descrição: Busca um paciente específico no FHIR pelo ID
    Parâmetros: fhir_id (string) - ID do paciente no FHIR
    Retorno: Dicionário com dados do paciente ou None se não encontrado
    """
    try:
        url = f"{FHIR_SERVER_URL}/Patient/{fhir_id}"
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()
        return resposta.json()
    except Exception as e:
        print(f"❌ Erro ao buscar paciente: {e}")
        return None

def atualizar_paciente_fhir(fhir_id: str, family_name: str, given_name: str,
                           gender: str, birth_date: str, phone: str,
                           city: str, state: str, postal_code: str) -> bool:
    """
    Descrição: Atualiza os dados de um paciente no FHIR
    Parâmetros: fhir_id, family_name, given_name, gender, birth_date, phone, city, state, postal_code
    Retorno: True se atualizado com sucesso, False caso contrário
    """
    try:
        payload = {
            "resourceType": "Patient",
            "id": fhir_id,
            "name": [
                {
                    "use": "official",
                    "family": family_name,
                    "given": [given_name]
                }
            ],
            "gender": gender,
            "birthDate": birth_date,
            "telecom": [
                {
                    "system": "phone",
                    "value": phone
                }
            ],
            "address": [
                {
                    "city": city,
                    "state": state,
                    "postalCode": postal_code
                }
            ]
        }

        url = f"{FHIR_SERVER_URL}/Patient/{fhir_id}"
        resposta = requests.put(url, json=payload, timeout=10)
        resposta.raise_for_status()
        print(f"✅ Paciente {fhir_id} atualizado no FHIR com sucesso!")
        return True
    except Exception as e:
        print(f"❌ Erro ao atualizar paciente no FHIR: {e}")
        return False

In [ ]:
# ===== CÉLULA 5: FUNÇÕES FHIR PARA OBSERVATIONS =====
# Funções para trabalhar com Observations (observações/sinais vitais)

def buscar_observations_por_paciente(patient_id: str) -> List[Dict]:
    """
    Descrição: Busca todas as observations de um paciente no FHIR
    Parâmetros: patient_id (string) - ID do paciente no FHIR
    Retorno: Lista de dicionários com as observations encontradas ou None se não encontrado
    """
    try:
        # Monta URL de busca usando o subject do paciente
        url = f"{FHIR_SERVER_URL}/Observation?subject=Patient/{patient_id}"
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()

        dados = resposta.json()
        observations = []

        # Extrai as observations da resposta
        if 'entry' in dados:
            for entrada in dados['entry']:
                recurso = entrada['resource']
                observations.append(recurso)

        return observations
    except Exception as e:
        print(f"❌ Erro ao buscar observations: {e}")
        return []

def criar_observation_fhir(patient_id: str, observation_type: str, value: float,
                          unit: str, effective_date_time: str) -> Optional[str]:
    """
    Descrição: Cria uma nova observation (sinal vital) no FHIR
    Parâmetros: patient_id, observation_type, value, unit, effective_date_time
    Retorno: ID da observation criada no FHIR ou None se erro
    """
    try:
        # Define o LOINC code baseado no tipo de observation
        loinc_codes = {
            "Pressão Sistólica": "8480-6",
            "Pressão Diastólica": "8462-4",
            "Frequência Cardíaca": "8867-4",
            "Temperatura": "8310-5",
            "Oxigenação": "59408-5"
        }

        # Caso observation_type nao seja encontrado no loinc_codes retorna o sengundo valor ("8867-4")
        loinc_code = loinc_codes.get(observation_type, "8867-4")

        # Monta o corpo da requisição seguindo o padrão FHIR
        payload = {
            "resourceType": "Observation",
            "status": "final",
            "category": [
                {
                    "coding": [
                        {
                            "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                            "code": "vital-signs",
                            "display": "Vital Signs"
                        }
                    ]
                }
            ],
            "code": {
                "coding": [
                    {
                        "system": "http://loinc.org",
                        "code": loinc_code,
                        "display": observation_type
                    }
                ],
                "text": observation_type
            },
            "subject": {
                "reference": f"Patient/{patient_id}"
            },
            "effectiveDateTime": effective_date_time,
            "valueQuantity": {
                "value": value,
                "unit": unit
            }
        }

        # Faz a requisição POST para criar a observation
        url = f"{FHIR_SERVER_URL}/Observation"
        resposta = requests.post(url, json=payload, timeout=10)
        resposta.raise_for_status()

        dados = resposta.json()
        fhir_id = dados.get('id')
        print(f"✅ Observation criada no FHIR com ID: {fhir_id}")
        return fhir_id
    except Exception as e:
        print(f"❌ Erro ao criar observation no FHIR: {e}")
        return None

def atualizar_observation_fhir(observation_id: str, value: float,
                              effective_date_time: str) -> bool:
    """
    Descrição: Atualiza uma observation existente no FHIR
    Parâmetros: observation_id, value, effective_date_time
    Retorno: True se atualizado com sucesso, False caso contrário
    """
    try:
        # Primeiro, busca a observation atual para manter os dados
        url = f"{FHIR_SERVER_URL}/Observation/{observation_id}"
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()
        observation_atual = resposta.json()

        # Atualiza apenas o valor e a data
        observation_atual['valueQuantity']['value'] = value
        observation_atual['effectiveDateTime'] = effective_date_time

        # Faz a requisição PUT para atualizar
        resposta = requests.put(url, json=observation_atual, timeout=10)
        resposta.raise_for_status()
        print(f"✅ Observation {observation_id} atualizada no FHIR com sucesso!")
        return True
    except Exception as e:
        print(f"❌ Erro ao atualizar observation no FHIR: {e}")
        return False

In [ ]:
# ===== CÉLULA 6: FUNÇÕES DE SINCRONIZAÇÃO SQL =====
# Funções para sincronizar dados entre FHIR e SQLite

def sincronizar_paciente_sql(fhir_id: str, paciente_dados: Dict) -> bool:
    """
    Descrição: Insere ou atualiza um paciente no banco de dados SQLite
    Parâmetros: fhir_id (string), paciente_dados (dicionário com dados do paciente)
    Retorno: True se sincronizado com sucesso, False caso contrário
    """
    try:
        conexao = obter_conexao()
        cursor = conexao.cursor()

        # Extrai dados do paciente
        family_name = paciente_dados.get('name', [{}])[0].get('family', '')
        given_name = paciente_dados.get('name', [{}])[0].get('given', [''])[0]
        gender = paciente_dados.get('gender', '')
        birth_date = paciente_dados.get('birthDate', '')

        # Extrai telefone
        phone = ''
        for telecom in paciente_dados.get('telecom', []):
            if telecom.get('system') == 'phone':
                phone = telecom.get('value', '')
                break

        # Extrai endereço
        city = ''
        state = ''
        postal_code = ''
        for address in paciente_dados.get('address', []):
            city = address.get('city', '')
            state = address.get('state', '')
            postal_code = address.get('postalCode', '')
            break

        # Verifica se paciente já existe
        cursor.execute('SELECT id FROM patients WHERE fhir_id = ?', (fhir_id,))
        # Retorna a primeira "linha" encontrada *fetchone()
        paciente_existente = cursor.fetchone()

        if paciente_existente:
            # Atualiza paciente existente
            cursor.execute('''
                UPDATE patients SET family_name = ?, given_name = ?, gender = ?,
                birth_date = ?, phone = ?, city = ?, state = ?, postal_code = ?
                WHERE fhir_id = ?
            ''', (family_name, given_name, gender, birth_date, phone, city, state, postal_code, fhir_id))
        else:
            # Insere novo paciente
            cursor.execute('''
                INSERT INTO patients (fhir_id, family_name, given_name, gender, birth_date, phone, city, state, postal_code)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (fhir_id, family_name, given_name, gender, birth_date, phone, city, state, postal_code))

        conexao.commit()
        conexao.close()
        return True
    except Exception as e:
        print(f"❌ Erro ao sincronizar paciente no SQL: {e}")
        return False

def sincronizar_observation_sql(fhir_id: str, patient_fhir_id: str, observation_dados: Dict) -> bool:
    """
    Descrição: Insere ou atualiza uma observation no banco de dados SQLite
    Parâmetros: fhir_id, patient_fhir_id, observation_dados
    Retorno: True se sincronizado com sucesso, False caso contrário
    """
    try:
        conexao = obter_conexao()
        cursor = conexao.cursor()

        # Extrai dados da observation
        observation_type = observation_dados.get('code', {}).get('text', '')
        value = observation_dados.get('valueQuantity', {}).get('value')
        unit = observation_dados.get('valueQuantity', {}).get('unit', '')
        effective_date_time = observation_dados.get('effectiveDateTime', '')

        # Verifica se observation já existe
        cursor.execute('SELECT id FROM observations WHERE fhir_id = ?', (fhir_id,))
        observation_existente = cursor.fetchone()

        if observation_existente:
            # Atualiza observation existente
            cursor.execute('''
                UPDATE observations SET observation_type = ?, value = ?, unit = ?, effective_date_time = ?
                WHERE fhir_id = ?
            ''', (observation_type, value, unit, effective_date_time, fhir_id))
        else:
            # Insere nova observation
            cursor.execute('''
                INSERT INTO observations (fhir_id, patient_fhir_id, observation_type, value, unit, effective_date_time)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (fhir_id, patient_fhir_id, observation_type, value, unit, effective_date_time))

        conexao.commit()
        conexao.close()
        return True
    except Exception as e:
        print(f"❌ Erro ao sincronizar observation no SQL: {e}")
        return False

In [ ]:
# ===== CÉLULA 7: FUNÇÕES DE CONSULTA SQL =====
# Funções para consultar dados no banco de dados SQLite

def obter_todos_pacientes() -> List[Dict]:
    try:
        conexao = obter_conexao()
        # Muda o formato do resultado que o banco retorna, de indice [0],[1]... para ['id'],['fhir_id']...
        conexao.row_factory = sqlite3.Row
        cursor = conexao.cursor()
        cursor.execute('SELECT * FROM patients')
        # Se fetchone é somente o primeiro valor, então fetchall são... isso todos
        pacientes = [dict(row) for row in cursor.fetchall()]
        conexao.close()
        return pacientes
    except Exception as e:
        print(f"❌ Erro ao obter pacientes: {e}")
        return []

def obter_todas_observations() -> List[Dict]:
    try:
        conexao = obter_conexao()
        conexao.row_factory = sqlite3.Row
        cursor = conexao.cursor()
        cursor.execute('SELECT * FROM observations')
        observations = [dict(row) for row in cursor.fetchall()]
        conexao.close()
        return observations
    except Exception as e:
        print(f"❌ Erro ao obter observations: {e}")
        return []

def obter_observations_por_paciente(patient_fhir_id: str) -> List[Dict]:
    try:
        conexao = obter_conexao()
        conexao.row_factory = sqlite3.Row
        cursor = conexao.cursor()
        cursor.execute('SELECT * FROM observations WHERE patient_fhir_id = ?', (patient_fhir_id,))
        observations = [dict(row) for row in cursor.fetchall()]
        conexao.close()
        return observations
    except Exception as e:
        print(f"❌ Erro ao obter observations do paciente: {e}")
        return []

def obter_paciente_por_fhir_id(fhir_id: str) -> Optional[Dict]:
    try:
        conexao = obter_conexao()
        conexao.row_factory = sqlite3.Row
        cursor = conexao.cursor()
        cursor.execute('SELECT * FROM patients WHERE fhir_id = ?', (fhir_id,))
        paciente = cursor.fetchone()
        conexao.close()
        return dict(paciente) if paciente else None
    except Exception as e:
        print(f"❌ Erro ao obter paciente: {e}")
        return None

In [ ]:
# ===== CÉLULA 8: FUNÇÕES DE EXIBIÇÃO DE DADOS =====
# Funções para exibir dados de forma formatada e legível
# ATUALIZADO: Mostra TODAS as colunas incluindo Estado, CEP e ID local

def exibir_paciente(paciente: dict):
    fhir_id = paciente.get('fhir_id') or paciente.get('id') or 'N/A'
    if 'name' in paciente and paciente['name']:
        family_name = paciente.get('name', [{}])[0].get('family') or 'N/A'
        given_list = paciente.get('name', [{}])[0].get('given') or []
        given_name = given_list[0] if given_list else 'N/A'
    else:
        family_name = paciente.get('family_name') or 'N/A'
        given_name = paciente.get('given_name') or 'N/A'
    gender = paciente.get('gender') or 'N/A'
    birth_date = paciente.get('birth_date') or paciente.get('birthDate') or 'N/A'
    phone = paciente.get('phone')
    if not phone and 'telecom' in paciente and paciente['telecom']:
        for t in paciente.get('telecom', []):
            if t.get('system') == 'phone':
                phone = t.get('value')
                break
    phone = phone or 'N/A'
    city = paciente.get('city')
    state = paciente.get('state')
    postal_code = paciente.get('postal_code')
    if 'address' in paciente and paciente['address']:
        addr = paciente.get('address', [{}])[0]
        if not city:
            city = addr.get('city')
        if not state:
            state = addr.get('state')
        if not postal_code:
            postal_code = addr.get('postalCode')
    city = city or 'N/A'
    state = state or 'N/A'
    postal_code = postal_code or 'N/A'
    print()
    print('=' * 60)
    print('📋 DADOS DO PACIENTE')
    print('=' * 60)
    print(f'ID FHIR: {fhir_id}')
    print(f'Nome Completo: {given_name} {family_name}')
    print(f'Gênero: {gender}')
    print(f'Data de Nascimento: {birth_date}')
    print(f'Telefone: {phone}')
    print(f'Cidade: {city}')
    print(f'Estado: {state}')
    print(f'CEP: {postal_code}')
    print('=' * 60)
    print()

def exibir_observation(observation: dict):
    fhir_id = observation.get('fhir_id') or observation.get('id') or 'N/A'
    obs_type = observation.get('observation_type')
    if not obs_type and 'code' in observation and observation['code']:
        obs_type = observation.get('code', {}).get('text')
    obs_type = obs_type or 'N/A'
    value = observation.get('value')
    unit = observation.get('unit')
    if 'valueQuantity' in observation and observation['valueQuantity']:
        vq = observation.get('valueQuantity', {})
        if value is None:
            value = vq.get('value')
        if not unit:
            unit = vq.get('unit')
    value_str = str(value) if value is not None else 'N/A'
    unit = unit or ''
    effective_dt = observation.get('effective_date_time') or observation.get('effectiveDateTime') or 'N/A'
    print(f'  ID FHIR: {fhir_id}')
    print(f'  Tipo: {obs_type}')
    print(f'  Valor: {value_str} {unit}')
    print(f'  Data/Hora: {effective_dt}')
    print('  ' + '-' * 50)

def exibir_tabela_pacientes(pacientes: list):
    print()
    print('=' * 200)
    print('📋 TABELA DE PACIENTES (COM TODAS AS COLUNAS)')
    print('=' * 200)
    if not pacientes:
        print('Nenhum paciente encontrado no banco de dados.')
    else:
        # Cabeçalho com ID local, ID FHIR, Nome, Gênero, Nascimento, Telefone, Cidade, Estado, CEP
        print(f'{"#ID":<6} {"ID FHIR":<18} {"Nome Completo":<30} {"Gênero":<8} {"Nascimento":<14} {"Telefone":<15} {"Cidade":<15} {"Estado":<12} {"CEP":<12}')
        print('-' * 200)
        for p in pacientes:
            # ID local do SQLite (comentado no código original, mas exibido aqui)
            local_id = p.get('id') or 'N/A'
            fhir_id = p.get('fhir_id') or 'N/A'
            nome = f"{p.get('given_name') or 'N/A'} {p.get('family_name') or 'N/A'}"
            gender = p.get('gender') or 'N/A'
            birth = p.get('birth_date') or 'N/A'
            phone = p.get('phone') or 'N/A'
            city = p.get('city') or 'N/A'
            # Estado e CEP (NOVAS COLUNAS)
            state = p.get('state') or 'N/A'
            postal = p.get('postal_code') or 'N/A'
            print(f'{str(local_id):<6} {fhir_id:<18} {nome:<30} {gender:<8} {birth:<14} {phone:<15} {city:<15} {state:<12} {postal:<12}')
    print('=' * 200)
    print()

def exibir_tabela_observations(observations: list):
    print()
    print('=' * 210)
    print('📋 TABELA DE OBSERVATIONS (COM TODAS AS COLUNAS)')
    print('=' * 210)
    if not observations:
        print('Nenhuma observation encontrada no banco de dados.')
    else:
        # Cabeçalho com ID local, ID FHIR, Paciente ID, Tipo, Valor, Unidade, Data/Hora
        print(f'{"#ID":<6} {"ID FHIR":<18} {"Paciente ID":<18} {"Tipo":<25} {"Valor":<12} {"Unidade":<10} {"Data/Hora":<35}')
        print('-' * 210)
        for obs in observations:
            # ID local do SQLite (comentado no código original, mas exibido aqui)
            local_id = obs.get('id') or 'N/A'
            fhir_id = obs.get('fhir_id') or 'N/A'
            patient_id = obs.get('patient_fhir_id') or 'N/A'
            obs_type = obs.get('observation_type') or 'N/A'
            value = obs.get('value')
            value_str = str(value) if value is not None else 'N/A'
            unit = obs.get('unit') or 'N/A'
            dt = obs.get('effective_date_time') or 'N/A'
            print(f'{str(local_id):<6} {fhir_id:<18} {patient_id:<18} {obs_type:<25} {value_str:<12} {unit:<10} {dt:<35}')
    print('=' * 210)
    print()


In [ ]:
# ===== CÉLULA 9: MENU BUSCAR =====
def funcao_buscar():
    while True:
        print()
        print('=' * 60)
        print('🔍 MENU BUSCAR')
        print('=' * 60)
        print('1 - Buscar Paciente por Nome')
        print('2 - Buscar Observations de um Paciente')
        print('0 - Voltar ao Menu Principal')
        print('=' * 60)
        opcao = input('Escolha uma opção: ').strip()
        if opcao == '1':
            nome = input('Digite o nome do paciente a buscar: ').strip()
            print()
            print('⏳ Buscando no FHIR...')
            pacientes_encontrados = buscar_pacientes_por_nome(nome)
            if pacientes_encontrados:
                print(f'✅ {len(pacientes_encontrados)} paciente(s) encontrado(s)!')
                for paciente in pacientes_encontrados:
                    fhir_id = paciente.get('id')
                    sincronizar_paciente_sql(fhir_id, paciente)
                    exibir_paciente(paciente)
                print('✅ Todos os pacientes foram sincronizados no SQLite!')
            else:
                print('❌ Nenhum paciente encontrado.')
        elif opcao == '2':
            patient_id = input('Digite o ID FHIR do paciente: ').strip()
            print()
            print('⏳ Buscando paciente no FHIR...')
            paciente = buscar_paciente_por_id_fhir(patient_id)
            if paciente:
                sincronizar_paciente_sql(patient_id, paciente)
                exibir_paciente(paciente)
                print('⏳ Buscando observations no FHIR...')
                observations = buscar_observations_por_paciente(patient_id)
                if observations:
                    print(f'✅ {len(observations)} observation(s) encontrada(s)!')
                    for obs in observations:
                        obs_id = obs.get('id')
                        sincronizar_observation_sql(obs_id, patient_id, obs)
                        exibir_observation(obs)
                    print('✅ Todas as observations foram sincronizadas no SQLite!')
                else:
                    print('ℹ️  Este paciente não possui observations ainda.')
            else:
                print('❌ Paciente não encontrado no FHIR.')
        elif opcao == '0':
            break
        else:
            print('❌ Opção inválida! Tente novamente.')

In [ ]:
# ===== CÉLULA 10: MENU VISUALIZAR DADOS LOCAIS =====
def funcao_visualizar_dados_locais():
    while True:
        print()
        print('=' * 60)
        print('📊 VISUALIZAR DADOS LOCAIS')
        print('=' * 60)
        print('1 - Ver Todos os Pacientes')
        print('2 - Ver Todas as Observations')
        print('0 - Voltar ao Menu Principal')
        print('=' * 60)
        opcao = input('Escolha uma opção: ').strip()
        if opcao == '1':
            pacientes = obter_todos_pacientes()
            exibir_tabela_pacientes(pacientes)
        elif opcao == '2':
            observations = obter_todas_observations()
            exibir_tabela_observations(observations)
        elif opcao == '0':
            break
        else:
            print('❌ Opção inválida! Tente novamente.')

In [ ]:
# ===== CÉLULA 11: MENU CRIAR PACIENTE =====
def funcao_criar_paciente():
    print()
    print('=' * 60)
    print('➕ CRIAR NOVO PACIENTE')
    print('=' * 60)
    family_name = input('Sobrenome: ').strip()
    given_name = input('Nome: ').strip()
    gender = input('Gênero (male/female/other): ').strip()
    birth_date = input('Data de Nascimento (YYYY-MM-DD): ').strip()
    phone = input('Telefone: ').strip()
    city = input('Cidade: ').strip()
    state = input('Estado: ').strip()
    postal_code = input('CEP: ').strip()
    print()
    print('⏳ Criando paciente no FHIR...')
    fhir_id = criar_paciente_fhir(family_name, given_name, gender, birth_date, phone, city, state, postal_code)
    if fhir_id:
        paciente = buscar_paciente_por_id_fhir(fhir_id)
        if paciente:
            if sincronizar_paciente_sql(fhir_id, paciente):
                print('✅ Paciente sincronizado no SQLite com sucesso!')
                exibir_paciente(paciente)
            else:
                print('❌ Erro ao sincronizar paciente no SQLite.')
        else:
            print('❌ Erro ao buscar dados do paciente criado.')
    else:
        print('❌ Erro ao criar paciente no FHIR.')

In [ ]:
# ===== CÉLULA 12: MENU CRIAR OBSERVATION =====
def funcao_criar_observation():
    print()
    print('=' * 60)
    print('➕ CRIAR NOVA OBSERVATION')
    print('=' * 60)
    patient_id = input('Digite o ID FHIR do paciente: ').strip()
    print()
    print('⏳ Buscando dados do paciente...')
    paciente = buscar_paciente_por_id_fhir(patient_id)
    if not paciente:
        print('❌ Paciente não encontrado no FHIR.')
        return
    sincronizar_paciente_sql(patient_id, paciente)
    exibir_paciente(paciente)
    tipos_mapeamento = {
        '1': ('Pressão Sistólica', 'mmHg'),
        '2': ('Pressão Diastólica', 'mmHg'),
        '3': ('Frequência Cardíaca', 'bpm'),
        '4': ('Temperatura', '°C'),
        '5': ('Oxigenação', '%'),
    }
    criar_mais = True
    while criar_mais:
        print()
        print('=' * 60)
        print('📋 SELECIONE O TIPO DE TRIAGEM')
        print('=' * 60)
        print('1 - Pressão Sistólica (mmHg)')
        print('2 - Pressão Diastólica (mmHg)')
        print('3 - Frequência Cardíaca (bpm)')
        print('4 - Temperatura (°C)')
        print('5 - Oxigenação/SpO2 (%)')
        print('0 - Voltar ao menu anterior')
        print('=' * 60)
        tipo_opcao = input('Escolha o tipo de triagem: ').strip()
        if tipo_opcao == '0':
            criar_mais = False
            break
        if tipo_opcao not in tipos_mapeamento:
            print('❌ Opção inválida!')
            continue
        observation_type, unit = tipos_mapeamento[tipo_opcao]
        valor_str = input(f'Digite o valor ({observation_type}): ').strip()
        try:
            valor = float(valor_str)
        except ValueError:
            print('❌ Valor inválido! Digite um número.')
            continue
        effective_date_time = datetime.now().isoformat()
        print()
        print('⏳ Criando observation no FHIR...')
        obs_fhir_id = criar_observation_fhir(patient_id, observation_type, valor, unit, effective_date_time)
        if not obs_fhir_id:
            print('❌ Erro ao criar observation no FHIR.')
            continue
        url = f'{FHIR_SERVER_URL}/Observation/{obs_fhir_id}'
        try:
            resposta = requests.get(url, timeout=10)
            resposta.raise_for_status()
            observation = resposta.json()
            if sincronizar_observation_sql(obs_fhir_id, patient_id, observation):
                print('✅ Observation sincronizada no SQLite com sucesso!')
                exibir_observation(observation)
            else:
                print('❌ Erro ao sincronizar observation no SQLite.')
        except Exception as e:
            print(f'❌ Erro ao buscar observation criada: {e}')
        resposta_usuario = input('\nDeseja criar outra observation para este paciente? (s/n): ').strip().lower()
        if resposta_usuario != 's':
            criar_mais = False

In [ ]:
# ===== CÉLULA 13: MENU ATUALIZAR PACIENTE =====
def funcao_atualizar_paciente():
    print()
    print('=' * 60)
    print('✏️ ATUALIZAR PACIENTE')
    print('=' * 60)
    patient_id = input('Digite o ID FHIR do paciente: ').strip()
    print()
    print('⏳ Buscando dados do paciente...')
    paciente = buscar_paciente_por_id_fhir(patient_id)
    if not paciente:
        print('❌ Paciente não encontrado no FHIR.')
        return
    sincronizar_paciente_sql(patient_id, paciente)
    print('\n📋 DADOS ATUAIS DO PACIENTE:')
    exibir_paciente(paciente)
    nome_atual = paciente.get('name', [{}])[0] if paciente.get('name') else {}
    family_atual = nome_atual.get('family') or ''
    given_list = nome_atual.get('given') or []
    given_atual = given_list[0] if given_list else ''
    gender_atual = paciente.get('gender') or ''
    birth_atual = paciente.get('birthDate') or ''
    phone_atual = ''
    for t in paciente.get('telecom', []):
        if t.get('system') == 'phone':
            phone_atual = t.get('value') or ''
            break
    addr_atual = (paciente.get('address') or [{}])[0] if paciente.get('address') else {}
    city_atual = addr_atual.get('city') or ''
    state_atual = addr_atual.get('state') or ''
    cep_atual = addr_atual.get('postalCode') or ''
    print()
    print('=' * 60)
    print('NOVOS DADOS (deixe em branco para manter o anterior)')
    print('=' * 60)
    family_name = input('Novo Sobrenome: ').strip() or family_atual
    given_name = input('Novo Nome: ').strip() or given_atual
    gender = input('Novo Gênero (male/female/other): ').strip() or gender_atual
    birth_date = input('Nova Data de Nascimento (YYYY-MM-DD): ').strip() or birth_atual
    phone = input('Novo Telefone: ').strip() or phone_atual
    city = input('Nova Cidade: ').strip() or city_atual
    state = input('Novo Estado: ').strip() or state_atual
    postal_code = input('Novo CEP: ').strip() or cep_atual
    print()
    print('⏳ Atualizando paciente no FHIR...')
    if atualizar_paciente_fhir(patient_id, family_name, given_name, gender, birth_date, phone, city, state, postal_code):
        paciente_atualizado = buscar_paciente_por_id_fhir(patient_id)
        if paciente_atualizado and sincronizar_paciente_sql(patient_id, paciente_atualizado):
            print('✅ Paciente sincronizado no SQLite com sucesso!')
            exibir_paciente(paciente_atualizado)
        else:
            print('❌ Erro ao sincronizar paciente no SQLite.')
    else:
        print('❌ Erro ao atualizar paciente no FHIR.')

In [ ]:
# ===== CÉLULA 14: MENU ATUALIZAR OBSERVATION =====
def funcao_atualizar_observation():
    print()
    print('=' * 60)
    print('✏️ ATUALIZAR OBSERVATION')
    print('=' * 60)
    patient_id = input('Digite o ID FHIR do paciente: ').strip()
    print()
    print('⏳ Buscando dados do paciente...')
    paciente = buscar_paciente_por_id_fhir(patient_id)
    if not paciente:
        print('❌ Paciente não encontrado no FHIR.')
        return
    sincronizar_paciente_sql(patient_id, paciente)
    exibir_paciente(paciente)
    print('⏳ Buscando observations no FHIR...')
    observations = buscar_observations_por_paciente(patient_id)
    if not observations:
        print('❌ Este paciente não possui observations.')
        return
    print('\n📋 OBSERVATIONS DO PACIENTE:')
    for i, obs in enumerate(observations, 1):
        obs_id = obs.get('id')
        sincronizar_observation_sql(obs_id, patient_id, obs)
        print(f'\n{i}. ')
        exibir_observation(obs)
    try:
        indice = int(input(f'\nQual observation deseja atualizar? (1-{len(observations)}): ')) - 1
        if indice < 0 or indice >= len(observations):
            print('❌ Índice inválido!')
            return
    except ValueError:
        print('❌ Entrada inválida!')
        return
    observation_selecionada = observations[indice]
    obs_id = observation_selecionada.get('id')
    print()
    print('=' * 60)
    print('Inserir novos dados')
    print('=' * 60)
    valor_str = input(f'Novo valor: ').strip()
    try:
        valor = float(valor_str)
    except ValueError:
        print('❌ Valor inválido! Digite um número.')
        return
    effective_date_time = datetime.now().isoformat()
    print()
    print('⏳ Atualizando observation no FHIR...')
    if atualizar_observation_fhir(obs_id, valor, effective_date_time):
        url = f'{FHIR_SERVER_URL}/Observation/{obs_id}'
        try:
            resposta = requests.get(url, timeout=10)
            resposta.raise_for_status()
            observation_atualizada = resposta.json()
            if sincronizar_observation_sql(obs_id, patient_id, observation_atualizada):
                print('✅ Observation sincronizada no SQLite com sucesso!')
                exibir_observation(observation_atualizada)
            else:
                print('❌ Erro ao sincronizar observation no SQLite.')
        except Exception as e:
            print(f'❌ Erro ao buscar observation atualizada: {e}')
    else:
        print('❌ Erro ao atualizar observation no FHIR.')

In [ ]:
# ===== CÉLULA 15: MENU PRINCIPAL =====
def menu_principal():
    while True:
        print()
        print('=' * 60)
        print('🏥 INTERFACE FHIR - SQLite')
        print('=' * 60)
        print('1 - 🔍 Buscar')
        print('2 - 📊 Visualizar dados locais')
        print('3 - ➕ Criar Paciente')
        print('4 - ➕ Criar Observation')
        print('5 - ✏️ Atualizar Paciente')
        print('6 - ✏️ Atualizar Observation')
        print('0 - ❌ Sair')
        print('=' * 60)
        opcao = input('Escolha uma opção: ').strip()
        if opcao == '1':
            funcao_buscar()
        elif opcao == '2':
            funcao_visualizar_dados_locais()
        elif opcao == '3':
            funcao_criar_paciente()
        elif opcao == '4':
            funcao_criar_observation()
        elif opcao == '5':
            funcao_atualizar_paciente()
        elif opcao == '6':
            funcao_atualizar_observation()
        elif opcao == '0':
            print()
            print('=' * 60)
            print('👋 Encerrando aplicação...')
            print('=' * 60)
            print()
            break
        else:
            print()
            print('❌ Opção inválida! Tente novamente.')

In [ ]:
menu_principal()


🏥 INTERFACE FHIR - SQLite
1 - 🔍 Buscar
2 - 📊 Visualizar dados locais
3 - ➕ Criar Paciente
4 - ➕ Criar Observation
5 - ✏️ Atualizar Paciente
6 - ✏️ Atualizar Observation
0 - ❌ Sair

🔍 MENU BUSCAR
1 - Buscar Paciente por Nome
2 - Buscar Observations de um Paciente
0 - Voltar ao Menu Principal

⏳ Buscando no FHIR...
✅ 1 paciente(s) encontrado(s)!

📋 DADOS DO PACIENTE
ID FHIR: 53517479
Nome Completo: Dalva Empresa
Gênero: other
Data de Nascimento: 1985-01-01
Telefone: N/A
Cidade: N/A
Estado: N/A
CEP: N/A

✅ Todos os pacientes foram sincronizados no SQLite!

🔍 MENU BUSCAR
1 - Buscar Paciente por Nome
2 - Buscar Observations de um Paciente
0 - Voltar ao Menu Principal

🏥 INTERFACE FHIR - SQLite
1 - 🔍 Buscar
2 - 📊 Visualizar dados locais
3 - ➕ Criar Paciente
4 - ➕ Criar Observation
5 - ✏️ Atualizar Paciente
6 - ✏️ Atualizar Observation
0 - ❌ Sair

➕ CRIAR NOVA OBSERVATION

⏳ Buscando dados do paciente...

📋 DADOS DO PACIENTE
ID FHIR: 53517479
Nome Completo: Dalva Empresa
Gênero: other
Data d